# Huấn luyện nhận dạng biển báo giao thông Việt Nam
**Đề tài:** Xây dựng ứng dụng trợ giúp người học nhận dạng biển báo giao thông sử dụng mạng nơron nhân tạo

Hệ thống 2 tầng:
1. **Phát hiện** (YOLO, 1 lớp "biển báo"): tự tìm vị trí biển trong ảnh/video, thay cho khoanh vùng thủ công.
2. **Phân loại** (CNN tự xây, huấn luyện bằng **lan truyền ngược**): xác định mã biển theo QCVN 41:2024/BGTVT.

Dữ liệu huấn luyện là **ảnh chụp thực tế**; tập kiểm thử lấy từ **nguồn độc lập** (không trộn vào huấn luyện, đã lọc ảnh gần trùng).

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`. Chạy lần lượt từng ô từ trên xuống.

## 1. Kết nối Google Drive và cài thư viện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, json
DRIVE   = '/content/drive/MyDrive/nhandang_bienbao'   # moi ket qua luu o day (khong mat khi Colab ngat)
NGUON   = '/content/nguon'                            # du lieu goc tai ve
DU_LIEU = '/content/du_lieu'                          # du lieu da gop + chia tap
PHIEN_BAN = 'v2'   # doi ten moi lan train lai -> giu ket qua lan truoc de so sanh trong thuyet minh
MO_HINH = f'{DRIVE}/3_mo_hinh'; KET_QUA = f'{DRIVE}/4_ket_qua_{PHIEN_BAN}'
for d in (DRIVE, NGUON, MO_HINH, KET_QUA): os.makedirs(d, exist_ok=True)
!pip -q install ultralytics roboflow kagglehub onnx onnxruntime
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ghi mã nguồn xử lý dữ liệu và huấn luyện

In [ ]:
%%writefile /content/xu_ly_du_lieu.py
# -*- coding: utf-8 -*-
"""
xu_ly_du_lieu.py - Gop nhieu bo du lieu bien bao (dinh dang YOLO) thanh:
  (1) bo du lieu PHAT HIEN: 1 lop "bien_bao" cho YOLO
  (2) bo du lieu PHAN LOAI: anh cat tung bien, moi thu muc la 1 ma QCVN 41:2024
Cac buoc: quet nguon -> anh xa nhan -> khu trung lap -> chia tap -> cat anh.
Tap KIEM THU lay tu NGUON DOC LAP (khong tron vao huan luyen), tranh loi
"train/test cung nguon" ma giao vien da nhan xet.
"""
import os, re, csv, json, glob, random, shutil, unicodedata
from collections import defaultdict, Counter
import numpy as np
import cv2
import yaml

DUOI_ANH = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
PAD = 0.15          # le them quanh hop bao khi cat bien (PHAI giong luc nhan dang)
CO_ANH_PL = 64      # kich thuoc anh dau vao bo phan loai
CANH_MIN = 12       # bo hop bao nho hon 12 px (qua nho de nhan dang)
LOP_NEN = "KHONG_PHAI_BIEN"

# ---------------------------------------------------------------- anh xa nhan
MA_RE = re.compile(r'(?<![A-Za-z])(DP|[PRWIS])[\s._\-]*(\d{3})([a-z](?![a-z]))?(?!\d)', re.I)

# Chi ghi nhung ten chac chan; ten khac se dua vao anh_xa_nhan.csv de nguoi dung tu dien.
TU_DIEN = {
    "no entry": "P.102", "cam di nguoc chieu": "P.102",
    "no passing": "P.125", "cam vuot": "P.125",
    "stop": "P.122", "dung lai": "P.122",
    "no stop or parking": "P.130", "no stopping and parking": "P.130", "cam dung va do xe": "P.130",
    "no parking": "P.131a", "cam do xe": "P.131a",
    "no vehicles": "P.101", "duong cam": "P.101",
    "vehicle no turn right": "P.103b", "vehicle no turn left": "P.103c",
    "no turn left": "P.123a", "no left turn": "P.123a", "cam re trai": "P.123a",
    "no turn right": "P.123b", "no right turn": "P.123b", "cam re phai": "P.123b",
    "pedestrians (warning)": "W.224", "children crossing": "W.225", "tre em": "W.225",
}
TU_BO = ("traffic light", "den giao thong", "person", "car", "motorbike", "vehicle_obj")


def bo_dau(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace('đ', 'd').replace('Đ', 'D')


def chuan_hoa_ten(s):
    return re.sub(r'[\s_\-]+', ' ', bo_dau(s).lower()).strip()


def doan_ma(ten_lop):
    """Tra ve (ma, cach) voi cach = 'ma_trong_ten' | 'tu_dien' | 'bo' | ''."""
    m = MA_RE.search(str(ten_lop))
    if m:
        ma = f"{m.group(1).upper()}.{m.group(2)}{(m.group(3) or '').lower()}"
        if ma == "P.127":  # bien toc do: tach theo so km/h
            con = MA_RE.sub(' ', str(ten_lop))
            v = re.search(r'(?<!\d)(\d{2,3})(?!\d)', con)
            if v:
                ma += f"({int(v.group(1))})"
        return ma, "ma_trong_ten"
    k = chuan_hoa_ten(ten_lop)
    if k in TU_DIEN:
        return TU_DIEN[k], "tu_dien"
    v = re.search(r'(?<!\d)(\d{2,3})(?!\d)', k)
    if v and any(t in k for t in ("speed limit", "toc do toi da", "gioi han toc do", "max speed")):
        return f"P.127({int(v.group(1))})", "tu_dien"
    if k in TU_BO:
        return "BO", "bo"
    return "", ""


# ---------------------------------------------------------------- doc nguon YOLO
def doc_ten_lop(goc):
    for p in sorted(glob.glob(os.path.join(goc, '**', '*.y*ml'), recursive=True)):
        try:
            d = yaml.safe_load(open(p, encoding='utf-8'))
        except Exception:
            continue
        if isinstance(d, dict) and 'names' in d:
            n = d['names']
            if isinstance(n, dict):
                return [str(n[k]) for k in sorted(n, key=lambda x: int(x))]
            return [str(x) for x in n]
    for ten in ('classes.txt', 'obj.names', '_darknet.labels', 'classes.names'):
        for p in glob.glob(os.path.join(goc, '**', ten), recursive=True):
            return [l.strip() for l in open(p, encoding='utf-8') if l.strip()]
    return None


def _tim_nhan(duong_anh):
    goc, _ = os.path.splitext(duong_anh)
    ung_vien = [goc + '.txt']
    parts = duong_anh.replace('\\', '/').split('/')
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].lower() in ('images', 'image', 'imgs', 'jpegimages'):
            q = parts[:i] + ['labels'] + parts[i + 1:]
            ung_vien.append(os.path.splitext('/'.join(q))[0] + '.txt')
            break
    for u in ung_vien:
        if os.path.isfile(u):
            return u
    return None


def doc_hop_bao(duong_nhan):
    """Doc nhan YOLO; ho tro ca dong polygon (chuyen thanh hop bao)."""
    hop = []
    for l in open(duong_nhan, encoding='utf-8', errors='ignore'):
        v = l.split()
        if len(v) < 5:
            continue
        try:
            c = int(float(v[0])); so = [float(x) for x in v[1:]]
        except ValueError:
            continue
        if len(so) == 4:
            cx, cy, w, h = so
        else:  # polygon x1 y1 x2 y2 ...
            xs, ys = so[0::2], so[1::2]
            cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
            w, h = max(xs) - min(xs), max(ys) - min(ys)
        if w > 0 and h > 0:
            hop.append((c, cx, cy, w, h))
    return hop


def quet_nguon(ten_nguon, goc):
    """Tra ve dict: ten_lop (list|None), anh: list(duong_anh, hop_bao)."""
    ten_lop = doc_ten_lop(goc)
    anh = []
    for p in glob.glob(os.path.join(goc, '**', '*'), recursive=True):
        if not p.lower().endswith(DUOI_ANH):
            continue
        n = _tim_nhan(p)
        if n is None:
            continue
        anh.append((p, doc_hop_bao(n)))
    anh.sort(key=lambda t: t[0])
    return {"nguon": ten_nguon, "goc": goc, "ten_lop": ten_lop, "anh": anh}


# ---------------------------------------------------------------- bang anh xa
def tao_bang_anh_xa(ds_nguon, duong_csv, thu_muc_mau, so_mau=6):
    """Ghi anh_xa_nhan.csv (nguon, id_lop, ten_lop, so_hop, ma_qcvn, cach) va anh mau cho lop chua ro."""
    cu = {}
    if os.path.isfile(duong_csv):  # giu lai phan nguoi dung da sua
        for r in csv.DictReader(open(duong_csv, encoding='utf-8-sig')):
            cu[(r['nguon'], r['id_lop'])] = r
    dem = Counter(); mau = defaultdict(list)
    for ng in ds_nguon:
        for p, hop in ng['anh']:
            for h in hop:
                k = (ng['nguon'], str(h[0])); dem[k] += 1
                if len(mau[k]) < so_mau:
                    mau[k].append((p, h))
    dong = []
    for ng in ds_nguon:
        ids = sorted({int(k[1]) for k in dem if k[0] == ng['nguon']})
        for i in ids:
            k = (ng['nguon'], str(i))
            ten = ng['ten_lop'][i] if ng['ten_lop'] and i < len(ng['ten_lop']) else f"lop_{i}"
            if k in cu:  # da co trong bang -> giu nguyen phan nguoi dung da sua
                ma, cach = cu[k]['ma_qcvn'], cu[k].get('cach', '')
            else:
                ma, cach = doan_ma(ten)
            dong.append({"nguon": k[0], "id_lop": k[1], "ten_lop": ten, "so_hop": dem[k],
                         "ma_qcvn": ma, "cach": cach})
            if not ma:  # luu anh mau de nguoi dung nhin va dien ma
                d = os.path.join(thu_muc_mau, f"{k[0]}__{i}__{re.sub(r'[^A-Za-z0-9]+', '_', bo_dau(ten))[:40]}")
                os.makedirs(d, exist_ok=True)
                for j, (p, h) in enumerate(mau[k]):
                    img = cv2.imread(p)
                    if img is not None:
                        c = cat_bien(img, h[1:])
                        if c is not None:
                            cv2.imwrite(os.path.join(d, f"{j}.png"), cv2.resize(c, (128, 128)))
    with open(duong_csv, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=list(dong[0].keys()) if dong else ["nguon"])
        w.writeheader(); w.writerows(dong)
    return dong


def doc_bang_anh_xa(duong_csv):
    """(nguon,id) -> ma. Dong do nguoi dung sua: dat cach='nguoi_dung' hoac chi can sua ma_qcvn."""
    bang = {}
    for r in csv.DictReader(open(duong_csv, encoding='utf-8-sig')):
        ma = (r.get('ma_qcvn') or '').strip()
        if ma.upper() in ('BO', 'KHAC'):
            ma = ma.upper()
        elif ma:
            ma = doan_ma(ma)[0] or ma
        bang[(r['nguon'], r['id_lop'])] = ma
    return bang


# ---------------------------------------------------------------- cat anh
def cat_bien(img, xywh, pad=PAD):
    H, W = img.shape[:2]
    cx, cy, w, h = xywh
    x1, x2 = (cx - w / 2) * W, (cx + w / 2) * W
    y1, y2 = (cy - h / 2) * H, (cy + h / 2) * H
    if min(x2 - x1, y2 - y1) < CANH_MIN:
        return None
    px, py = (x2 - x1) * pad, (y2 - y1) * pad
    x1, y1 = int(max(0, x1 - px)), int(max(0, y1 - py))
    x2, y2 = int(min(W, x2 + px)), int(min(H, y2 + py))
    if x2 - x1 < 4 or y2 - y1 < 4:
        return None
    return img[y1:y2, x1:x2]


# ---------------------------------------------------------------- khu trung lap
def dhash(duong):
    g = cv2.imread(duong, cv2.IMREAD_GRAYSCALE)
    if g is None:
        return None
    g = cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return np.packbits(bits).view('>u8')[0].astype(np.uint64)


def _popcount(x):
    if hasattr(np, 'bitwise_count'):
        return np.bitwise_count(x)
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1).reshape(x.shape)


def gan_trung(h_kt, h_hl, nguong=6, khoi=512):
    """Mask: anh kiem thu nao gan trung (Hamming <= nguong) voi bat ky anh huan luyen nao."""
    h_kt = np.asarray(h_kt, dtype=np.uint64); h_hl = np.asarray(h_hl, dtype=np.uint64)
    trung = np.zeros(len(h_kt), bool)
    for i in range(0, len(h_kt), khoi):
        d = _popcount(np.bitwise_xor(h_kt[i:i + khoi, None], h_hl[None, :]))
        trung[i:i + khoi] = (d <= nguong).any(1)
    return trung


# ---------------------------------------------------------------- xay dung bo du lieu
def xay_dung(ds_nguon, bang, nguon_kiem_thu, ra, ti_le_val=0.12, khoi_lien_tiep=40,
             min_mau=30, toi_da_moi_lop=3000, seed=0, log=print):
    """
    - nguon_kiem_thu: list ten nguon dung lam tap KIEM THU DOC LAP (vd ['charlesmld']).
    - Cac nguon con lai chia train/val theo KHOI anh lien tiep (giam ro ri khung hinh video lien ke).
    Ket qua:
      ra/phat_hien/{images,labels}/{train,val,test} + data.yaml   (YOLO 1 lop)
      ra/phan_loai/{train,val,test}/<ma>/*.png                       (anh cat 64x64)
      ra/thong_ke.json, ra/loai_bo.csv
    """
    rnd = random.Random(seed)
    if os.path.isdir(ra):
        shutil.rmtree(ra)
    loai_bo = []  # (duong, ly_do)

    # 1) hash + bo trung lap tuyet doi trong tap huan luyen
    muc = []  # (nguon, duong, hop, hash)
    for ng in ds_nguon:
        for p, hop in ng['anh']:
            h = dhash(p)
            if h is None:
                loai_bo.append((p, 'khong_doc_duoc_anh')); continue
            muc.append((ng['nguon'], p, hop, h))
    kt = [m for m in muc if m[0] in nguon_kiem_thu]
    hl_all = [m for m in muc if m[0] not in nguon_kiem_thu]
    da_thay, hl = set(), []
    for m in hl_all:
        if int(m[3]) in da_thay:
            loai_bo.append((m[1], 'trung_lap_trong_tap_huan_luyen')); continue
        da_thay.add(int(m[3])); hl.append(m)

    # 2) bo anh kiem thu gan trung voi anh huan luyen (dam bao doc lap)
    if kt and hl:
        tr = gan_trung([m[3] for m in kt], [m[3] for m in hl])
        for m, t in zip(kt, tr):
            if t:
                loai_bo.append((m[1], 'kiem_thu_gan_trung_anh_huan_luyen'))
        kt = [m for m, t in zip(kt, tr) if not t]

    # 3) chia train/val theo khoi lien tiep trong tung nguon
    tach = {}
    theo_nguon = defaultdict(list)
    for m in hl:
        theo_nguon[m[0]].append(m)
    for ng, ds in theo_nguon.items():
        # nhom anh cung goc: Roboflow dat ten anh tang cuong la <goc>.rf.<ma>.jpg
        nhom = defaultdict(list)
        for m in ds:
            nhom[os.path.basename(m[1]).split('.rf.')[0]].append(m)
        khoi, cur = [], []
        for k in sorted(nhom):
            cur += nhom[k]
            if len(cur) >= khoi_lien_tiep:
                khoi.append(cur); cur = []
        if cur:
            khoi.append(cur)
        rnd.shuffle(khoi)
        n_val = int(round(len(khoi) * ti_le_val)) if len(khoi) >= 8 else (1 if len(khoi) >= 3 else 0)
        for j, k in enumerate(khoi):
            for m in k:
                tach[m[1]] = 'val' if j < n_val else 'train'
    for m in kt:
        tach[m[1]] = 'test'
    tat_ca = hl + kt

    # 4) bo du lieu phat hien (1 lop)
    for s in ('train', 'val', 'test'):
        os.makedirs(f"{ra}/phat_hien/images/{s}", exist_ok=True)
        os.makedirs(f"{ra}/phat_hien/labels/{s}", exist_ok=True)
    tk_det = Counter()
    for i, (ng, p, hop, _) in enumerate(tat_ca):
        s = tach[p]
        dong = []
        for (c, cx, cy, w, h) in hop:
            ma = bang.get((ng, str(c)), '')
            if ma == 'BO':
                continue
            dong.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        ten = f"{ng}_{i:06d}"
        dich = f"{ra}/phat_hien/images/{s}/{ten}{os.path.splitext(p)[1].lower()}"
        try:
            os.link(p, dich)
        except OSError:
            shutil.copy2(p, dich)
        open(f"{ra}/phat_hien/labels/{s}/{ten}.txt", 'w').write('\n'.join(dong))
        tk_det[s] += 1
    with open(f"{ra}/phat_hien/data.yaml", 'w') as f:
        yaml.safe_dump({"path": os.path.abspath(f"{ra}/phat_hien"), "train": "images/train",
                        "val": "images/val", "test": "images/test", "names": {0: "bien_bao"}}, f)

    # 5) bo du lieu phan loai (anh cat)
    cat = defaultdict(list)   # (split, ma) -> list anh
    nen = defaultdict(list)
    for i, (ng, p, hop, _) in enumerate(tat_ca):
        s = tach[p]
        img = cv2.imread(p)
        if img is None:
            continue
        H, W = img.shape[:2]
        hop_px = []
        for j, (c, cx, cy, w, h) in enumerate(hop):
            hop_px.append(((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
            ma = bang.get((ng, str(c)), '')
            if not ma or ma in ('BO', 'KHAC'):
                continue
            cr = cat_bien(img, (cx, cy, w, h))
            if cr is None:
                continue
            cat[(s, ma)].append((f"{ng}_{i:06d}_{j}", cv2.resize(cr, (CO_ANH_PL, CO_ANH_PL), interpolation=cv2.INTER_AREA)))
        # 1 manh nen ngau nhien / anh (khong chong len bien nao) -> lop KHONG_PHAI_BIEN
        if hop_px and rnd.random() < 0.5:
            ws = [b[2] - b[0] for b in hop_px]
            for _ in range(10):
                kc = int(np.clip(rnd.choice(ws) * rnd.uniform(0.8, 2.0), 20, min(H, W) - 1))
                x, y = rnd.randint(0, W - kc), rnd.randint(0, H - kc)
                if all(_iou((x, y, x + kc, y + kc), b) < 0.02 for b in hop_px):
                    nen[s].append((f"{ng}_{i:06d}_nen", cv2.resize(img[y:y + kc, x:x + kc], (CO_ANH_PL, CO_ANH_PL), interpolation=cv2.INTER_AREA)))
                    break

    dem_train = Counter({ma: len(v) for (s, ma), v in cat.items() if s == 'train'})
    lop_giu = sorted([ma for ma, n in dem_train.items() if n >= min_mau])
    so_nen = int(np.median([min(dem_train[m], toi_da_moi_lop) for m in lop_giu])) if lop_giu else 0
    tk_pl = defaultdict(dict)
    for (s, ma), ds in cat.items():
        if ma not in lop_giu:
            # bien NGOAI danh sach: tu nguon kiem thu -> do kha nang tu choi (bao cao);
            # tu nguon huan luyen -> dung de CHON NGUONG (khong cham vao tap test)
            _ghi(f"{ra}/phan_loai/{'ngoai_danh_sach' if s == 'test' else 'ngoai_danh_sach_val'}/{ma}", ds)
            continue
        if s == 'train' and len(ds) > toi_da_moi_lop:
            rnd.shuffle(ds); ds = ds[:toi_da_moi_lop]
        _ghi(f"{ra}/phan_loai/{s}/{ma}", ds); tk_pl[s][ma] = len(ds)
    for s, ds in nen.items():
        rnd.shuffle(ds)
        k = so_nen if s == 'train' else int(so_nen * len(ds) / max(1, len(nen.get('train', []))))
        ds = ds[:max(k, 1)]
        _ghi(f"{ra}/phan_loai/{s}/{LOP_NEN}", ds); tk_pl[s][LOP_NEN] = len(ds)

    with open(f"{ra}/loai_bo.csv", 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f); w.writerow(['duong_anh', 'ly_do']); w.writerows(loai_bo)
    tk = {"so_anh_phat_hien": dict(tk_det), "so_anh_cat_phan_loai": {s: dict(sorted(v.items())) for s, v in tk_pl.items()},
          "so_lop_phan_loai": len(lop_giu) + 1, "lop_bi_bo_vi_it_mau(<%d)" % min_mau: sorted(set(dem_train) - set(lop_giu)),
          "ly_do_loai_bo": dict(Counter(r[1] for r in loai_bo)), "nguon_kiem_thu": list(nguon_kiem_thu)}
    json.dump(tk, open(f"{ra}/thong_ke.json", 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    log(json.dumps({k: v for k, v in tk.items() if k != "so_anh_cat_phan_loai"}, ensure_ascii=False, indent=2))
    return tk


def _ghi(d, ds):
    os.makedirs(d, exist_ok=True)
    for ten, im in ds:
        cv2.imwrite(os.path.join(d, ten + '.png'), im)


def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i
    return i / u if u > 0 else 0.0


# ---------------------------------------------------------------- anh tu chup
def nap_anh_tu_chup(goc):
    """Anh tu chup: goc/<ma>/*.jpg (1 bien chinh/anh, khong can khoanh hop) -> dung cho danh gia DAU-CUOI."""
    ds = []
    for d in sorted(glob.glob(os.path.join(goc, '*'))):
        if os.path.isdir(d):
            for p in sorted(glob.glob(os.path.join(d, '*'))):
                if p.lower().endswith(DUOI_ANH):
                    ds.append((p, os.path.basename(d).strip()))
    return ds

In [ ]:
%%writefile /content/huan_luyen_phan_loai.py
# -*- coding: utf-8 -*-
"""
huan_luyen_phan_loai.py - Huan luyen mang no-ron tich chap (CNN) phan loai bien bao
bang thuat toan LAN TRUYEN NGUOC (Back Propagation) + toi uu AdamW.

Dau vao : <du_lieu>/phan_loai/{train,val,test}/<ma_bien>/*.png  (anh mau 64x64)
Dau ra  : <ra>/phan_loai.pt, phan_loai.onnx, lop.json, nguong.json,
          lich_su.csv, bieu_do_huan_luyen.png, bao_cao_<tap>.csv, ma_tran_nham_lan_<tap>.png,
          do_tin_cay.png, kiem_tra_gradient.txt, anh_sai_<tap>.png, tom_tat.json
"""
import os, json, csv, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CO_ANH = 64
LOP_NEN = "KHONG_PHAI_BIEN"


# ------------------------------------------------------------------ mo hinh
class MangCNN(nn.Module):
    """3 khoi [Conv3x3-BN-ReLU-Conv3x3-BN-ReLU-MaxPool] + 2 lop day du (FC). Anh mau 64x64x3."""
    def __init__(self, so_lop):
        super().__init__()
        def khoi(vao, ra):
            return nn.Sequential(
                nn.Conv2d(vao, ra, 3, padding=1, bias=False), nn.BatchNorm2d(ra), nn.ReLU(inplace=True),
                nn.Conv2d(ra, ra, 3, padding=1, bias=False), nn.BatchNorm2d(ra), nn.ReLU(inplace=True),
                nn.MaxPool2d(2))
        self.dac_trung = nn.Sequential(khoi(3, 32), khoi(32, 64), khoi(64, 128))   # 64 -> 8
        self.phan_loai = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4), nn.Linear(128 * 8 * 8, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.4), nn.Linear(256, so_lop))

    def forward(self, x):
        return self.phan_loai(self.dac_trung(x))


def bang_tham_so(model):
    dong, tong = [], 0
    for ten, p in model.named_parameters():
        dong.append((ten, tuple(p.shape), p.numel())); tong += p.numel()
    return dong, tong


# ------------------------------------------------------------------ du lieu
def bien_doi(train):
    chuan = [transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)]
    if not train:
        return transforms.Compose([transforms.Resize((CO_ANH, CO_ANH))] + chuan)
    # KHONG lat ngang: lat se doi nghia bien (cam re trai <-> cam re phai)
    return transforms.Compose([
        transforms.Resize((CO_ANH, CO_ANH)),
        transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.85, 1.15), shear=5),
        transforms.ColorJitter(0.4, 0.4, 0.3, 0.02),
        transforms.RandomApply([transforms.GaussianBlur(3, (0.1, 1.2))], p=0.25),
    ] + chuan + [transforms.RandomErasing(p=0.15, scale=(0.02, 0.1))])


def nap_tap(goc, lop, train=False):
    """Nap ImageFolder nhung ep thu tu lop theo danh sach `lop` cua tap train; bo lop la."""
    ds = datasets.ImageFolder(goc, transform=bien_doi(train))
    anh_xa = {i: lop.index(c) for c, i in ds.class_to_idx.items() if c in lop}
    ds.samples = [(p, anh_xa[y]) for p, y in ds.samples if y in anh_xa]
    ds.targets = [y for _, y in ds.samples]
    ds.classes, ds.class_to_idx = lop, {c: i for i, c in enumerate(lop)}
    return ds


# ------------------------------------------------------------------ kiem tra lan truyen nguoc
def kiem_tra_gradient(so_lop, so_diem=12, eps=1e-6, seed=0):
    """So sanh gradient tinh bang lan truyen nguoc (autograd) voi dao ham so (sai phan trung tam)."""
    torch.manual_seed(seed)
    m = MangCNN(so_lop).double().eval()          # eval: tat dropout, BN dung thong ke co dinh
    x = torch.randn(4, 3, CO_ANH, CO_ANH, dtype=torch.double)
    y = torch.randint(0, so_lop, (4,))
    m.zero_grad(); F.cross_entropy(m(x), y).backward()
    kq = []
    with torch.no_grad():
        for ten, p in m.named_parameters():
            if p.grad is None or p.numel() == 0:
                continue
            # chon cac phan tu co |gradient| lon de sai so tuong doi co y nghia
            ung_vien = torch.randint(0, p.numel(), (min(200, p.numel()),))
            chon = ung_vien[p.grad.view(-1)[ung_vien].abs().argsort(descending=True)[:max(1, so_diem // 6)]]
            for i in chon.tolist():
                goc = p.view(-1)[i].item()
                p.view(-1)[i] = goc + eps; lp = F.cross_entropy(m(x), y).item()
                p.view(-1)[i] = goc - eps; lm = F.cross_entropy(m(x), y).item()
                p.view(-1)[i] = goc
                so = (lp - lm) / (2 * eps); bp = p.grad.view(-1)[i].item()
                sai = abs(so - bp) / max(1e-12, abs(so) + abs(bp))
                kq.append((ten, i, bp, so, sai))
    return kq


# ------------------------------------------------------------------ hieu chuan (temperature scaling)
def tim_nhiet_do(logits, nhan):
    """Tim T > 0 lam cuc tieu NLL tren tap val (Guo et al., 2017)."""
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def dong():
        opt.zero_grad(); l = F.cross_entropy(logits / logT.exp(), nhan); l.backward(); return l
    opt.step(dong)
    return float(logT.exp())


def ece(xs, dung, so_ngan=15):
    """Expected Calibration Error."""
    bien = np.linspace(0, 1, so_ngan + 1); e = 0.0
    for a, b in zip(bien[:-1], bien[1:]):
        m = (xs > a) & (xs <= b)
        if m.any():
            e += m.mean() * abs(dung[m].mean() - xs[m].mean())
    return float(e)


# ------------------------------------------------------------------ du doan
@torch.no_grad()
def du_doan(model, loader, thiet_bi):
    model.eval(); L, Y = [], []
    for x, y in loader:
        L.append(model(x.to(thiet_bi)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)


def ve_ma_tran(cm, lop, duong, tieu_de):
    n = len(lop); kt = max(8, n * 0.28)
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(kt, kt))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(lop, rotation=90, fontsize=7); ax.set_yticklabels(lop, fontsize=7)
    ax.set_xlabel("Du doan"); ax.set_ylabel("Nhan dung"); ax.set_title(tieu_de)
    if n <= 30:
        for i in range(n):
            for j in range(n):
                if cm[i, j]:
                    ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6,
                            color="white" if cmn[i, j] > 0.5 else "black")
    fig.tight_layout(); fig.savefig(duong, dpi=130); plt.close(fig)


def luoi_anh_sai(ds, chi_so, du, lop, duong, toi_da=48):
    chi_so = list(chi_so)[:toi_da]
    if not chi_so:
        return
    c = 8; r = math.ceil(len(chi_so) / c)
    fig, axs = plt.subplots(r, c, figsize=(c * 1.6, r * 1.9))
    axs = np.array(axs).reshape(-1)
    for a in axs:
        a.axis("off")
    for a, i in zip(axs, chi_so):
        p, y = ds.samples[i]
        a.imshow(plt.imread(p)); a.set_title(f"{lop[y]}\n->{lop[du[i]]}", fontsize=6, color="red")
    fig.tight_layout(); fig.savefig(duong, dpi=120); plt.close(fig)


def danh_gia(model, ds, ten, ra, lop, T, nguong, thiet_bi, bs=256):
    if len(ds) == 0:
        return None
    L, Y = du_doan(model, DataLoader(ds, bs, num_workers=2), thiet_bi)
    P = F.softmax(L / T, 1); conf, du = P.max(1)
    y, du, conf = Y.numpy(), du.numpy(), conf.numpy()
    nhan_co = sorted(set(y) | set(du))
    bc = classification_report(y, du, labels=nhan_co, target_names=[lop[i] for i in nhan_co],
                               output_dict=True, zero_division=0)
    with open(f"{ra}/bao_cao_{ten}.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f); w.writerow(["lop", "precision", "recall", "f1", "so_mau"])
        for k, v in bc.items():
            if isinstance(v, dict):
                w.writerow([k, f"{v['precision']:.4f}", f"{v['recall']:.4f}", f"{v['f1-score']:.4f}", int(v['support'])])
    cm = confusion_matrix(y, du, labels=list(range(len(lop))))
    ve_ma_tran(cm, lop, f"{ra}/ma_tran_nham_lan_{ten}.png", f"Ma tran nham lan - tap {ten} (n={len(y)})")
    sai = np.where(du != y)[0]
    luoi_anh_sai(ds, sai, du, lop, f"{ra}/anh_sai_{ten}.png")
    # cap nham lan nhieu nhat
    cap = sorted([(int(cm[i, j]), lop[i], lop[j]) for i in range(len(lop)) for j in range(len(lop)) if i != j and cm[i, j]], reverse=True)[:15]
    nhan = conf >= nguong
    kq = {"tap": ten, "so_mau": int(len(y)), "so_dung": int((du == y).sum()),
          "accuracy": float((du == y).mean()),
          "macro_f1": float(f1_score(y, du, labels=np.unique(y), average="macro", zero_division=0)),
          "so_lop_co_trong_tap": int(len(np.unique(y))),
          "recall_tung_lop": {lop[i]: round(float(cm[i, i] / cm[i].sum()), 4) for i in np.unique(y)},
          "ty_le_chap_nhan_o_nguong": float(nhan.mean()),
          "accuracy_tren_mau_chap_nhan": float((du[nhan] == y[nhan]).mean()) if nhan.any() else None,
          "ece": ece(conf, (du == y).astype(float)), "cap_nham_lan_nhieu_nhat": cap,
          "lop_f1_duoi_0.9": sorted([(k, round(v["f1-score"], 3), int(v["support"])) for k, v in bc.items()
                                     if isinstance(v, dict) and k in lop and v["support"] > 0 and v["f1-score"] < 0.9],
                                    key=lambda t: t[1])}
    return kq


def danh_gia_tu_choi(model, goc, lop, T, nguong, thiet_bi):
    """Anh bien NGOAI danh sach lop: ly tuong la do tin cay < nguong (bi tu choi)."""
    if not os.path.isdir(goc):
        return None
    ds = datasets.ImageFolder(goc, transform=bien_doi(False))
    if len(ds) == 0:
        return None
    L, _ = du_doan(model, DataLoader(ds, 256), thiet_bi)
    conf = F.softmax(L / T, 1).max(1).values.numpy()
    return {"so_mau_ngoai_danh_sach": int(len(conf)), "ty_le_bi_tu_choi": float((conf < nguong).mean()),
            "do_tin_cay": conf}


# ------------------------------------------------------------------ chuong trinh chinh
def huan_luyen(du_lieu, ra, so_epoch_toi_da=80, kien_nhan=10, bs=128, lr=1e-3, seed=0,
               ty_le_chap_nhan_toi_thieu=0.95, log=print):
    os.makedirs(ra, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    thiet_bi = "cuda" if torch.cuda.is_available() else "cpu"
    goc = f"{du_lieu}/phan_loai"
    lop = sorted(os.listdir(f"{goc}/train"))
    tr, va = nap_tap(f"{goc}/train", lop, True), nap_tap(f"{goc}/val", lop)
    te = nap_tap(f"{goc}/test", lop) if os.path.isdir(f"{goc}/test") else None
    log(f"Thiet bi: {thiet_bi} | so lop: {len(lop)} | train {len(tr)} | val {len(va)} | test {len(te) if te else 0}")

    # 1) kiem tra lan truyen nguoc truoc khi huan luyen
    kq_g = kiem_tra_gradient(len(lop))
    with open(f"{ra}/kiem_tra_gradient.txt", "w", encoding="utf-8") as f:
        f.write("tham_so\tchi_so\tgrad_lan_truyen_nguoc\tgrad_so\tsai_so_tuong_doi\n")
        for t in kq_g:
            f.write(f"{t[0]}\t{t[1]}\t{t[2]:.6e}\t{t[3]:.6e}\t{t[4]:.2e}\n")
    ss = np.array([t[4] for t in kq_g]); dat = int((ss < 1e-4).sum())
    with open(f"{ra}/kiem_tra_gradient.txt", "a", encoding="utf-8") as f:
        f.write(f"\n# {dat}/{len(ss)} diem co sai so tuong doi < 1e-4; trung vi = {np.median(ss):.2e}. "
                "Diem lech (neu co) do ham ReLU/MaxPool khong kha vi tai diem gay.\n")
    log(f"Kiem tra gradient: {dat}/{len(ss)} diem dat (<1e-4), trung vi sai so {np.median(ss):.2e}")

    # 2) can bang lop: lay mau theo trong so 1/sqrt(so_mau)
    dem = np.bincount(tr.targets, minlength=len(lop))
    w = 1.0 / np.sqrt(np.maximum(dem, 1))
    sampler = WeightedRandomSampler(w[tr.targets], num_samples=len(tr), replacement=True)
    dl_tr = DataLoader(tr, bs, sampler=sampler, num_workers=2, drop_last=True)
    dl_va = DataLoader(va, 256, num_workers=2)

    model = MangCNN(len(lop)).to(thiet_bi)
    ts, tong = bang_tham_so(model)
    with open(f"{ra}/so_tham_so.txt", "w", encoding="utf-8") as f:
        for t in ts:
            f.write(f"{t[0]}\t{t[1]}\t{t[2]}\n")
        f.write(f"TONG\t\t{tong}\n")
    log(f"Tong so tham so: {tong:,}")

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    tot_nhat, epoch_tot, cho = -1.0, 0, 0
    ls = []
    t0 = time.time()
    for ep in range(1, so_epoch_toi_da + 1):
        model.train(); tl, td, tn = 0.0, 0, 0
        for x, y in dl_tr:
            x, y = x.to(thiet_bi), y.to(thiet_bi)
            out = model(x)                      # lan truyen xuoi
            loss = F.cross_entropy(out, y)      # ham mat mat
            opt.zero_grad(); loss.backward()    # lan truyen nguoc: tinh dL/dw cho moi trong so
            opt.step()                          # cap nhat trong so
            tl += loss.item() * len(y); td += (out.argmax(1) == y).sum().item(); tn += len(y)
        Lv, Yv = du_doan(model, dl_va, thiet_bi)
        vl = F.cross_entropy(Lv, Yv).item(); dv = Lv.argmax(1)
        va_acc = (dv == Yv).float().mean().item()
        va_f1 = f1_score(Yv.numpy(), dv.numpy(), labels=np.unique(Yv.numpy()), average="macro", zero_division=0)
        sch.step(va_f1)
        ls.append([ep, tl / tn, td / tn, vl, va_acc, va_f1, opt.param_groups[0]["lr"]])
        log(f"epoch {ep:3d} | loss {tl/tn:.4f} acc {td/tn:.4f} | val loss {vl:.4f} acc {va_acc:.4f} macroF1 {va_f1:.4f} | {time.time()-t0:.0f}s")
        if va_f1 > tot_nhat + 1e-4:
            tot_nhat, epoch_tot, cho = va_f1, ep, 0
            torch.save({"model": model.state_dict(), "lop": lop, "epoch": ep}, f"{ra}/phan_loai.pt")
        else:
            cho += 1
            if cho >= kien_nhan:   # DUNG SOM THUC SU: dung khi val macro-F1 khong tang sau `kien_nhan` epoch
                log(f"Dung som tai epoch {ep}; tot nhat o epoch {epoch_tot} (val macro-F1 {tot_nhat:.4f})")
                break

    with open(f"{ra}/lich_su.csv", "w", newline="") as f:
        c = csv.writer(f); c.writerow(["epoch", "loss", "acc", "val_loss", "val_acc", "val_macro_f1", "lr"]); c.writerows(ls)
    a = np.array(ls)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(a[:, 0], a[:, 1], label="train"); ax[0].plot(a[:, 0], a[:, 3], label="val"); ax[0].set_title("Ham mat mat"); ax[0].legend()
    ax[1].plot(a[:, 0], a[:, 2], label="train acc"); ax[1].plot(a[:, 0], a[:, 4], label="val acc"); ax[1].plot(a[:, 0], a[:, 5], label="val macro-F1")
    ax[1].axvline(epoch_tot, ls="--", c="gray"); ax[1].set_title("Do chinh xac"); ax[1].legend()
    for x in ax:
        x.set_xlabel("epoch")
    fig.tight_layout(); fig.savefig(f"{ra}/bieu_do_huan_luyen.png", dpi=130); plt.close(fig)

    # 3) nap trong so tot nhat, hieu chuan va chon nguong tren VAL (khong dung test)
    model.load_state_dict(torch.load(f"{ra}/phan_loai.pt", map_location=thiet_bi)["model"])
    Lv, Yv = du_doan(model, dl_va, thiet_bi)
    dung_goc = (Lv.argmax(1) == Yv).numpy().astype(float)
    ece_truoc = ece(F.softmax(Lv, 1).max(1).values.numpy(), dung_goc)
    T = tim_nhiet_do(Lv, Yv)
    ece_sau = ece(F.softmax(Lv / T, 1).max(1).values.numpy(), dung_goc)
    if ece_sau >= ece_truoc:      # hieu chuan khong cai thien ECE -> khong dung
        T = 1.0
    Pv = F.softmax(Lv / T, 1); cv, dv = Pv.max(1)
    dung_v = (dv == Yv).numpy(); cv = cv.numpy()
    tc_val = danh_gia_tu_choi(model, f"{goc}/ngoai_danh_sach_val", lop, T, 0.5, thiet_bi)  # bien la tu nguon huan luyen
    tc = danh_gia_tu_choi(model, f"{goc}/ngoai_danh_sach", lop, T, 0.5, thiet_bi)          # bien la tu nguon kiem thu (chi bao cao)
    # NGUONG = gia tri LON NHAT ma van chap nhan >= ty_le_chap_nhan_toi_thieu bien trong danh sach (tren val):
    # tu choi duoc nhieu bien la nhat ma khong tu choi qua nhieu bien quen.
    nguong = 0.30
    for t in np.round(np.arange(0.30, 0.995, 0.01), 2):
        if (cv >= t).mean() >= ty_le_chap_nhan_toi_thieu:
            nguong = float(t)
    with open(f"{ra}/bang_nguong.csv", "w", newline="") as f:   # can cu de chon nguong
        c = csv.writer(f)
        c.writerow(["nguong", "ty_le_chap_nhan_val", "do_chinh_xac_khi_chap_nhan_val",
                    "ty_le_tu_choi_bien_la_val", "ty_le_tu_choi_bien_la_test"])
        for t in sorted(set(np.round(np.arange(0.30, 1.0, 0.05), 2).tolist() + [nguong])):
            m = cv >= t
            c.writerow([t, f"{m.mean():.4f}", f"{dung_v[m].mean():.4f}" if m.any() else "",
                        f"{(tc_val['do_tin_cay'] < t).mean():.4f}" if tc_val else "",
                        f"{(tc['do_tin_cay'] < t).mean():.4f}" if tc else ""])
    for x in (tc_val, tc):
        if x:
            x["ty_le_bi_tu_choi"] = float((x["do_tin_cay"] < nguong).mean())
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(cv[dung_v], bins=40, alpha=.6, label="val - du doan dung", density=True)
    if (~dung_v).any():
        ax.hist(cv[~dung_v], bins=40, alpha=.6, label="val - du doan sai", density=True)
    if tc_val:
        ax.hist(tc_val["do_tin_cay"], bins=40, alpha=.5, label="bien la (nguon huan luyen)", density=True)
    ax.axvline(nguong, c="k", ls="--", label=f"nguong = {nguong:.2f}")
    ax.set_xlabel("Do tin cay (softmax)"); ax.legend(); fig.tight_layout()
    fig.savefig(f"{ra}/do_tin_cay.png", dpi=130); plt.close(fig)

    tom_tat = {"so_lop": len(lop), "tong_tham_so": tong, "epoch_tot_nhat": epoch_tot, "so_epoch_da_chay": len(ls),
               "nhiet_do_T": T, "ece_val_truoc_hieu_chuan": ece_truoc, "ece_val_sau_hieu_chuan": ece_sau,
               "dung_hieu_chuan": T != 1.0, "nguong": nguong,
               "quy_tac_chon_nguong": f"nguong lon nhat sao cho >= {ty_le_chap_nhan_toi_thieu:.0%} mau val duoc chap nhan"}
    for ten, ds in (("val", va), ("test", te)):
        if ds is not None:
            tom_tat[ten] = danh_gia(model, ds, ten, ra, lop, T, nguong, thiet_bi)
    if tc_val:
        tom_tat["tu_choi_bien_la_val"] = {k: v for k, v in tc_val.items() if k != "do_tin_cay"}
    if tc:
        tom_tat["tu_choi_bien_ngoai_danh_sach"] = {k: v for k, v in tc.items() if k != "do_tin_cay"}
    json.dump(lop, open(f"{ra}/lop.json", "w", encoding="utf-8"), ensure_ascii=False)
    json.dump({"nguong": nguong, "nhiet_do_T": T, "co_anh": CO_ANH, "pad": 0.15, "lop_nen": LOP_NEN,
               "chuan_hoa": "rgb/255, (x-0.5)/0.5"}, open(f"{ra}/nguong.json", "w"), indent=2)
    json.dump(tom_tat, open(f"{ra}/tom_tat.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    # 4) xuat ONNX de ung dung chay khong can PyTorch
    model.eval().cpu()
    x = torch.randn(1, 3, CO_ANH, CO_ANH)
    kw = dict(input_names=["anh"], output_names=["logits"], dynamic_axes={"anh": {0: "n"}, "logits": {0: "n"}}, opset_version=17)
    try:
        torch.onnx.export(model, x, f"{ra}/phan_loai.onnx", dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model, x, f"{ra}/phan_loai.onnx", **kw)
    log(json.dumps({k: v for k, v in tom_tat.items() if k not in ("val",)}, ensure_ascii=False, indent=1, default=str)[:3000])
    return tom_tat


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--du_lieu", required=True); ap.add_argument("--ra", required=True)
    ap.add_argument("--epoch", type=int, default=80); ap.add_argument("--kien_nhan", type=int, default=10)
    a = ap.parse_args()
    huan_luyen(a.du_lieu, a.ra, a.epoch, a.kien_nhan)

## 3. Tải dữ liệu ảnh thật
- **GitHub** CharlesMld: ~11.700 ảnh, 16 lớp, giấy phép MIT (mặc định dùng làm **tập kiểm thử độc lập**).
- **Kaggle**: nếu báo lỗi đăng nhập, chạy `kagglehub.login()` và dán username + key (kaggle.com → Settings → API → Create New Token).
- **Roboflow** (**bắt buộc** — lần 1 thiếu bộ này nên chỉ có ~4.000 ảnh train): tạo tài khoản miễn phí tại roboflow.com → Settings → API Keys, dán vào `ROBOFLOW_API_KEY`.
  Số phiên bản (version): mở trang bộ dữ liệu → *Download Dataset* → *YOLOv8* → *Show download code* và chép đúng số `version(...)`.

In [ ]:
# 3a. GitHub (khong can tai khoan, ~3,2 GB)
if not os.path.isdir(f'{NGUON}/charlesmld'):
    !git clone -q --depth 1 https://github.com/CharlesMld/vietnam_traffic_signs_dataset.git {NGUON}/charlesmld

In [ ]:
# 3b. Kaggle
import kagglehub
KAGGLE = {
    'kaggle_maitam': 'maitam/vietnamese-traffic-signs',                                  # ~3.200 anh, CC0
    'kaggle_jayden': 'jaydenguyenx/vietnamese-traffic-signs-detection-and-recognition',   # 1.170 anh, 29 lop
}
for ten, slug in KAGGLE.items():
    if os.path.isdir(f'{NGUON}/{ten}'): print(ten, 'da co'); continue
    try:
        p = kagglehub.dataset_download(slug)
        shutil.copytree(p, f'{NGUON}/{ten}'); print(ten, 'OK')
    except Exception as e:
        print(ten, 'LOI:', e)

In [ ]:
# 3c. Roboflow (them/bot bo du lieu tuy y; dinh dang: (workspace, project, version))
ROBOFLOW_API_KEY = ''   # <-- dan API key vao day
ROBOFLOW = [
    ('giang-yp9g1', 'vietnam-traffic-sign-altsi', 3),   # ~56 ma bien, CC BY 4.0
]
if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for ws, pj, v in ROBOFLOW:
        ten = 'rf_' + pj.replace('-', '_')
        if os.path.isdir(f'{NGUON}/{ten}'): print(ten, 'da co'); continue
        try:
            rf.workspace(ws).project(pj).version(v).download('yolov8', location=f'{NGUON}/{ten}'); print(ten, 'OK')
        except Exception as e:
            print(ten, 'LOI (kiem tra lai so version):', e)
else:
    print('Chua co ROBOFLOW_API_KEY -> bo qua Roboflow')

## 4. Quét dữ liệu và lập bảng ánh xạ nhãn → mã QCVN

In [ ]:
sys.path.insert(0, '/content')
import importlib, pandas as pd, xu_ly_du_lieu as X; importlib.reload(X)
ds_nguon = [X.quet_nguon(t, f'{NGUON}/{t}') for t in sorted(os.listdir(NGUON)) if os.path.isdir(f'{NGUON}/{t}')]
for d in ds_nguon:
    print(f"{d['nguon']:<28} {len(d['anh']):>6} anh | {len(d['ten_lop'] or [])} lop")
bang = pd.DataFrame(X.tao_bang_anh_xa(ds_nguon, f'{DRIVE}/anh_xa_nhan.csv', f'{DRIVE}/kiem_tra_nhan'))
pd.set_option('display.max_rows', 500)
print('\nCAC LOP CHUA CO MA (xem anh mau trong Drive/nhandang_bienbao/kiem_tra_nhan):')
display(bang[bang.ma_qcvn == ''])

### Bước làm tay (quan trọng, chỉ làm 1 lần)
1. Xem bảng trên và ảnh mẫu trong thư mục `kiem_tra_nhan` trên Drive.
2. Điền mã vào ô dưới theo dạng `('nguon', 'id_lop'): 'MÃ'`. Giá trị đặc biệt: `'KHAC'` = là biển nhưng không rõ mã (chỉ dùng cho bộ phát hiện), `'BO'` = không phải biển (đèn, xe...).
3. **Kiểm tra cả các mã được gán tự động** (cột `cach`) với QCVN 41:2024 — đặc biệt các biển đã đổi mã so với 2019.
4. Chạy ô dưới, sau đó chạy lại ô số 4 để xem bảng đã đủ.

In [ ]:
SUA = {
    # ('charlesmld', '8'): 'R.302a',
    # ('kaggle_maitam', '3'): 'KHAC',
}
df = pd.read_csv(f'{DRIVE}/anh_xa_nhan.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
for (ng, i), ma in SUA.items():
    m = (df.nguon == ng) & (df.id_lop == str(i)); df.loc[m, 'ma_qcvn'] = ma; df.loc[m, 'cach'] = 'nguoi_dung'
df.to_csv(f'{DRIVE}/anh_xa_nhan.csv', index=False, encoding='utf-8-sig')
display(df.groupby('ma_qcvn').so_hop.apply(lambda s: s.astype(int).sum()).sort_values(ascending=False).head(80))

## 5. Gộp dữ liệu, khử trùng lặp, chia tập
- `NGUON_KIEM_THU`: nguồn dùng làm **tập kiểm thử độc lập** (không đưa vào huấn luyện).
- Ảnh kiểm thử gần trùng với ảnh huấn luyện (dHash, Hamming ≤ 6) bị loại → ghi vào `loai_bo.csv`.
- Train/val chia theo **khối ảnh liên tiếp** (tránh 2 khung hình liền nhau của cùng video rơi vào 2 tập).
- Lớp có < `MIN_MAU` ảnh huấn luyện bị loại khỏi bộ phân loại (ghi rõ trong `thong_ke.json`).

In [ ]:
importlib.reload(X)
NGUON_KIEM_THU = ['charlesmld']
MIN_MAU = 30
bang = X.doc_bang_anh_xa(f'{DRIVE}/anh_xa_nhan.csv')
tk = X.xay_dung(ds_nguon, bang, NGUON_KIEM_THU, DU_LIEU, min_mau=MIN_MAU)
for f in ('thong_ke.json', 'loai_bo.csv'): shutil.copy(f'{DU_LIEU}/{f}', KET_QUA)
pd.DataFrame(tk['so_anh_cat_phan_loai']).fillna(0).astype(int)

## 6. Huấn luyện bộ PHÂN LOẠI (CNN, lan truyền ngược) — khoảng 15–40 phút

In [ ]:
import huan_luyen_phan_loai as H; importlib.reload(H)
tom_tat = H.huan_luyen(DU_LIEU, f'{KET_QUA}/phan_loai', so_epoch_toi_da=80, kien_nhan=10,
                       ty_le_chap_nhan_toi_thieu=0.95)
for f in ('phan_loai.onnx', 'phan_loai.pt', 'lop.json', 'nguong.json'):
    shutil.copy(f'{KET_QUA}/phan_loai/{f}', MO_HINH)

In [ ]:
from IPython.display import Image, display
for f in ('bieu_do_huan_luyen.png', 'ma_tran_nham_lan_test.png', 'do_tin_cay.png', 'anh_sai_test.png'):
    p = f'{KET_QUA}/phan_loai/{f}'
    if os.path.exists(p): print(f); display(Image(p, width=900))
t = tom_tat.get('test') or {}
print(f"TAP KIEM THU DOC LAP: dung {t.get('so_dung')}/{t.get('so_mau')} | accuracy {t.get('accuracy', 0):.4f} | macro-F1 {t.get('macro_f1', 0):.4f} ({t.get('so_lop_co_trong_tap')} lop)")
print('Recall tung lop:', t.get('recall_tung_lop'))
print('Nguong:', tom_tat['nguong'], '| tu choi bien la (val):', (tom_tat.get('tu_choi_bien_la_val') or {}).get('ty_le_bi_tu_choi'),
      '| tu choi bien la (test):', (tom_tat.get('tu_choi_bien_ngoai_danh_sach') or {}).get('ty_le_bi_tu_choi'))
print('Lop F1 < 0,9:', t.get('lop_f1_duoi_0.9'))

## 7. Huấn luyện bộ PHÁT HIỆN (YOLO11n) — 2–4 giờ, tự lưu vào Drive, chạy tiếp được nếu Colab ngắt

In [ ]:
# 7a. CHAY THU NHANH (~5 phut): 10% du lieu, 2 epoch. Neu o nay loi thi DUNG LAI sua truoc khi train that.
from ultralytics import YOLO
YOLO('yolo11n.pt').train(data=f'{DU_LIEU}/phat_hien/data.yaml', epochs=2, imgsz=640, batch=32, fraction=0.1,
                         project='/content/thu', name='thu', exist_ok=True, workers=2, plots=False)
print('CHAY THU OK -> chay o tiep theo de train that')

Nếu Colab bị ngắt: chạy lại ô 1, 2, 4, 5 (dữ liệu sinh lại giống hệt vì cố định seed) rồi chạy ô dưới — chương trình tự chạy tiếp từ `last.pt`.

In [ ]:
from ultralytics import YOLO
RUN = f'{DRIVE}/runs'; TEN_DET = f'phat_hien_{PHIEN_BAN}'
last = f'{RUN}/{TEN_DET}/weights/last.pt'
if os.path.exists(last) and not os.path.exists(f'{RUN}/{TEN_DET}/xong.txt'):
    m = YOLO(last); m.train(resume=True)
else:
    m = YOLO('yolo11n.pt')
    m.train(data=f'{DU_LIEU}/phat_hien/data.yaml', epochs=50, imgsz=640, batch=32, patience=10,
            project=RUN, name=TEN_DET, exist_ok=True, workers=2, seed=0)
open(f'{RUN}/{TEN_DET}/xong.txt', 'w').write('ok')

In [ ]:
best = YOLO(f'{RUN}/{TEN_DET}/weights/best.pt')
r = best.val(data=f'{DU_LIEU}/phat_hien/data.yaml', split='test', project=RUN, name=f'{TEN_DET}_kiem_thu', exist_ok=True)
kq_det = {'mAP50': float(r.box.map50), 'mAP50_95': float(r.box.map), 'precision': float(r.box.mp), 'recall': float(r.box.mr)}
print('BO PHAT HIEN tren tap kiem thu doc lap:', kq_det)
json.dump(kq_det, open(f'{KET_QUA}/phat_hien_kiem_thu.json', 'w'), indent=2)
p = best.export(format='onnx', imgsz=640, opset=12)
shutil.copy(p, f'{MO_HINH}/phat_hien.onnx'); print('Da luu', f'{MO_HINH}/phat_hien.onnx')

## 8. Đóng gói để tải về máy (D:\\nhandang\\bienbao)

In [ ]:
shutil.make_archive(f'{DRIVE}/tai_ve_3_mo_hinh', 'zip', MO_HINH)
shutil.make_archive(f'{DRIVE}/tai_ve_4_ket_qua_{PHIEN_BAN}', 'zip', KET_QUA)
print('Tai 2 file zip trong Google Drive/nhandang_bienbao ve may.')